# AnomalyCLIP fixed-perturbation evaluation
Enable a GPU and Internet, then attach only MVTec AD, VisA, and the generated perturbation dataset. The notebook clones this evaluator and official AnomalyCLIP automatically, discovers the mounted dataset paths, inventories both prompt folders, calibrates thresholds once from clean predictions, and evaluates every attack condition matching the selection cell.

In [ ]:
# ============================== USER SELECTION ==============================
TARGETS = ('mvtec', 'visa')
MVTEC_INPUT = '/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'
VISA_INPUT = '/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'
PERTURBATION_INPUT = '/kaggle/input/datasets/parsaorbot/perterbation-generated'
PROMPT_MODES = ('frozen_prompt', 'learnable_prompt')  # None means every available mode
SETUP_IDS = None  # e.g. ('steps500_eps2',); None means every available setup
SCOPES = ('per_dataset', 'per_category', 'per_image')
CATEGORIES = None
DIRECTIONS = None
LOSS_MODES = None
LOSS_FORMULATIONS = None
BATCH_SIZE = 2
SAVE_PREDICTIONS = False
OVERWRITE = False
MAX_CONDITIONS = None  # Set to 1 only for a plumbing check.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

WORKING = Path('/kaggle/working')
EVALUATOR_ROOT = WORKING / 'adversarial-perturbation-evaluator'
ANOMALYCLIP_ROOT = WORKING / 'AnomalyCLIP'
OUTPUT_ROOT = WORKING / 'fixed_perturbation_results'
EVALUATOR_URL = 'https://github.com/Parsagh05/adversarial-perturbation-evaluator.git'
ANOMALYCLIP_URL = 'https://github.com/zqhang/AnomalyCLIP.git'

def clone_or_update(url, destination):
    if (destination / '.git').is_dir():
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(destination)], check=True)

clone_or_update(EVALUATOR_URL, EVALUATOR_ROOT)
clone_or_update(ANOMALYCLIP_URL, ANOMALYCLIP_ROOT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    f'{EVALUATOR_ROOT}[anomalyclip]',
], check=True)
print('Evaluator commit:', subprocess.check_output(
    ['git', '-C', str(EVALUATOR_ROOT), 'rev-parse', 'HEAD'], text=True
).strip())

In [ ]:
from fpeval.kaggle import discover_kaggle_inputs, inventory_attack_setups

mounted = discover_kaggle_inputs(
    '/kaggle/input',
    mvtec_root=MVTEC_INPUT,
    visa_root=VISA_INPUT,
    attacks_root=PERTURBATION_INPUT,
)
inventory = inventory_attack_setups(mounted.attacks_root)
print('MVTec root:', mounted.mvtec_root)
print('VisA root:', mounted.visa_root)
print('Perturbation root:', mounted.attacks_root)
print('Available attack setups:', json.dumps(inventory, indent=2))

selected_modes = tuple(PROMPT_MODES) if PROMPT_MODES is not None else tuple(inventory)
available_selected_modes = tuple(mode for mode in selected_modes if mode in inventory)
if not available_selected_modes:
    raise RuntimeError(f'None of the selected prompt modes are available: {selected_modes}')
if SETUP_IDS is not None:
    available_ids = {setup_id for mode in available_selected_modes for setup_id in inventory[mode]}
    missing_ids = sorted(set(SETUP_IDS) - available_ids)
    if missing_ids:
        raise RuntimeError(f'Selected setup IDs are not mounted: {missing_ids}')
print('Selected prompt modes present on disk:', available_selected_modes)
print('Selected setup IDs:', SETUP_IDS if SETUP_IDS is not None else 'all available')

In [ ]:
mvtec_checkpoint = ANOMALYCLIP_ROOT / 'checkpoints/9_12_4_multiscale/epoch_15.pth'
visa_checkpoint = ANOMALYCLIP_ROOT / 'checkpoints/9_12_4_multiscale_visa/epoch_15.pth'
for checkpoint in (mvtec_checkpoint, visa_checkpoint):
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Official AnomalyCLIP checkpoint missing: {checkpoint}')

all_model_kwargs = {
    'mvtec': {
        'repository': str(ANOMALYCLIP_ROOT),
        'checkpoint': str(mvtec_checkpoint),
        'download_root': str(WORKING / 'clip-cache'),
    },
    'visa': {
        'repository': str(ANOMALYCLIP_ROOT),
        'checkpoint': str(visa_checkpoint),
        'download_root': str(WORKING / 'clip-cache'),
    },
}
config = {
    'attacks_root': str(mounted.attacks_root),
    'output_root': str(OUTPUT_ROOT),
    'model': 'anomalyclip',
    'mvtec_root': str(mounted.mvtec_root),
    'visa_root': str(mounted.visa_root),
    'targets': list(TARGETS),
    'scopes': list(SCOPES),
    'prompt_modes': list(available_selected_modes),
    'setup_ids': list(SETUP_IDS) if SETUP_IDS is not None else None,
    'categories': list(CATEGORIES) if CATEGORIES is not None else None,
    'directions': list(DIRECTIONS) if DIRECTIONS is not None else None,
    'loss_modes': list(LOSS_MODES) if LOSS_MODES is not None else None,
    'loss_formulations': list(LOSS_FORMULATIONS) if LOSS_FORMULATIONS is not None else None,
    'model_kwargs_by_target': {target: all_model_kwargs[target] for target in TARGETS},
    'device': 'cuda',
    'batch_size': BATCH_SIZE,
    'image_size': 518,
    'gaussian_sigma': 4.0,
    'pixel_threshold_modes': ['fixed_0_5', 'image_f1', 'clean_pixel_f1'],
    # Qualitative selection uses the clean-calibrated pixel F1-max threshold.
    'qualitative_threshold_modes': ['clean_pixel_f1'],
    'verify_checksums': True,
    'save_predictions': SAVE_PREDICTIONS,
    'save_qualitative_samples': True,
    'write_separated_results': True,
    'create_output_archives': True,
    'overwrite': OVERWRITE,
    'max_conditions': MAX_CONDITIONS,
}
config_path = WORKING / 'anomalyclip.json'
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(config_path.read_text())

In [ ]:
subprocess.run([sys.executable, '-m', 'fpeval', '--config', str(config_path)], check=True)
model_output = OUTPUT_ROOT / 'anomalyclip'
threshold_path = model_output / 'thresholds.json'
print('Automatically calibrated and frozen thresholds:', threshold_path)
print(threshold_path.read_text()[:4000])
print('Downloadable output archives:')
for name in ('anomalyclip', 'anomalyclip_samples', 'anomalyclip_separated', 'anomalyclip_samples_separated'):
    print(' -', OUTPUT_ROOT / f'{name}.zip')